In [1]:
import torch
print("GPU var mı:", torch.cuda.is_available())

!pip install -q kaggle grad-cam
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d ham10000

GPU var mı: False
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:52<00:00, 107MB/s]



In [2]:
import os, glob, shutil
import pandas as pd
from google.colab import files

ESIK = 0.35
tp = pd.read_csv("test_tahmin.csv")
yollar = {os.path.splitext(os.path.basename(p))[0]: p
          for p in glob.glob("ham10000/**/*.jpg", recursive=True)}
tp["path"] = tp["image_id"].map(yollar)

def sec(maske, hedef):
    df = tp[maske].copy()
    return df.iloc[(df.prob - hedef).abs().argsort().iloc[0]]

secimler = {
    "1_melanom_tespit":      sec((tp.dx == "mel") & (tp.prob >= ESIK), 0.85),
    "2_bcc_tespit":          sec((tp.dx == "bcc") & (tp.prob >= ESIK), 0.80),
    "3_nevus_iyi_huylu":     sec((tp.dx == "nv") & (tp.prob < ESIK), 0.05),
    "4_keratoz_iyi_huylu":   sec((tp.dx == "bkl") & (tp.prob < ESIK), 0.15),
    "5_melanom_kacirilan":   sec((tp.dx == "mel") & (tp.prob < ESIK), 0.20),
    "6_yanlis_alarm":        sec((tp.label == 0) & (tp.prob >= ESIK), 0.60),
}

os.makedirs("ornekler", exist_ok=True)
for ad, r in secimler.items():
    shutil.copy(r.path, f"ornekler/{ad}.jpg")
    print(f"{ad:24s} {r.image_id}  {r.dx:5s}  olasılık = {r.prob:.2f}")

shutil.make_archive("ornekler", "zip", "ornekler")
files.download("ornekler.zip")

1_melanom_tespit         ISIC_0025132  mel    olasılık = 0.85
2_bcc_tespit             ISIC_0026350  bcc    olasılık = 0.80
3_nevus_iyi_huylu        ISIC_0034232  nv     olasılık = 0.05
4_keratoz_iyi_huylu      ISIC_0024877  bkl    olasılık = 0.15
5_melanom_kacirilan      ISIC_0028148  mel    olasılık = 0.20
6_yanlis_alarm           ISIC_0032691  nv     olasılık = 0.60


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>